In [1]:
import pandas as pd
import folium
import random

shapes = pd.read_csv("../data/rozklad/shapes.txt")
trips = pd.read_csv("../data/rozklad/trips.txt")
routes = pd.read_csv("../data/rozklad/routes.txt")

# shapes=shapes[shapes['shape_id']=='shape_6254']



In [2]:


merged = trips.merge(routes, on="route_id")[["shape_id", "route_short_name"]].drop_duplicates()
shapes = shapes.merge(merged, on="shape_id", how="left")

shapes=shapes.drop(columns='shape_dist_traveled')
shapes=shapes.dropna()
shapes['route_short_name']=shapes['route_short_name'].astype(int)

# shapes[(shapes['route_short_name']==102)&(shapes['shape_pt_sequence']==1)]

print(shapes['route_short_name'].loc[shapes.index[0]])

102


In [3]:


center_lat = shapes["shape_pt_lat"].mean()
center_lon = shapes["shape_pt_lon"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")

unique_routes = shapes["shape_id"].unique()


for route in unique_routes:
    subset = shapes[shapes["shape_id"] == route].sort_values("shape_pt_sequence")
    coords = subset[["shape_pt_lat", "shape_pt_lon"]].values.tolist()

    # losowy kolor (można też użyć route_color z GTFS)
    color = "#%06x" % random.randint(0, 0xFFFFFF)

    # Sprawdź, czy coords nie jest puste przed dodaniem PolyLine
    if coords:
        # print(shapes.loc[shapes['shape_id'] == route, 'route_short_name'].iloc[0],route)
        folium.PolyLine(
            locations=coords,
            color=color,
            weight=random.randint(2,7),
            opacity=0.9,
            tooltip=f"Linia {shapes.loc[shapes['shape_id'] == route, 'route_short_name'].iloc[0]}"
        ).add_to(m)
    else:
        print(f"Warning: No coordinates for route {route}, skipping PolyLine.")
# 5. Zapisz mapę do pliku HTML
m.save("linie_autobusowe.html")